In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import yfinance as yf
import datetime
import xgboost

from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import minimize, least_squares, newton, differential_evolution, dual_annealing
from scipy.special import ndtr

from sklearn.model_selection import train_test_split
from sklearn.metrics import explained_variance_score, mean_absolute_percentage_error, mean_squared_error, mean_squared_log_error
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.metrics import r2_score
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
from scipy.optimize import brentq
from scipy.stats.qmc import LatinHypercube

from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from joblib import Parallel, delayed
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from itertools import repeat
import random
import xgboost as xgb_lib
import os as _os
import json as _json

In [15]:
# Load the created Stata files

path = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\data_output_final\data_output_2003_top3_qual.dta"
data = pd.read_stata(path)
data = data.drop(["index"], axis = 1)


In [16]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# GPU-compatible Heston pricer
def heston_charf(phi, S0, V0, rf, lambd, tau, kappa, theta, sigma, rho):
    phi = phi.to(torch.complex64)

    a = kappa * theta
    b = kappa + lambd

    d = torch.sqrt((rho * sigma * phi * 1j - b)**2
        + sigma**2 * (phi**2 + 1j * phi) + 0j)

    g = (b - rho * sigma * phi * 1j - d) / (b - rho * sigma * phi * 1j + d)

    exp_dt = torch.exp(-d * tau)

    C = (
        rf * phi * 1j * tau
        + a / sigma**2 * (
            (b - rho * sigma * phi * 1j - d) * tau
            - 2 * torch.log((1 - g * exp_dt) / (1 - g))
        )
    )

    D = (
        (b - rho * sigma * phi * 1j - d) / sigma**2
        * (1 - exp_dt) / (1 - g * exp_dt)
    )

    log_cf = C + D * V0 + 1j * phi * torch.log(S0)

    log_cf = torch.clamp(log_cf.real, -100.0, 50.0) + 1j * log_cf.imag
    return torch.exp(log_cf)


def integrand(phi, S0, K, V0, rf, lambd, tau, kappa, theta, sigma, rho):
    arg = (S0, V0, rf, lambd, tau, kappa, theta, sigma, rho)

    phi = phi.to(torch.complex64)

    num = (
        torch.exp(rf * tau) * heston_charf(phi - 1j, *arg)
        - K * heston_charf(phi, *arg)
    )

    den = 1j * phi * K**(1j * phi)

    return num / den





_GL_N = 64
_gl_nodes_np, _gl_weights_np = np.polynomial.laguerre.laggauss(_GL_N)

def heston_price_gpu(S0, K, V0, rf, lambd, tau, kappa, theta, sigma, rho, n=128):
    
    batch_n = S0.shape[0]

    device = torch.device('cuda')
    S0 = torch.as_tensor(S0, device=device, dtype=torch.float64)
    K = torch.as_tensor(K, device=device, dtype=torch.float64)
    V0 = torch.as_tensor(V0, device=device, dtype=torch.float64)
    rf = torch.as_tensor(rf, device=device, dtype=torch.float64)
    lambd = torch.as_tensor(lambd, device=device, dtype=torch.float64)
    tau = torch.as_tensor(tau, device=device, dtype=torch.float64)
    kappa = torch.as_tensor(kappa, device=device, dtype=torch.float64)
    theta = torch.as_tensor(theta, device=device, dtype=torch.float64)
    sigma = torch.as_tensor(sigma, device=device, dtype=torch.float64)
    rho = torch.as_tensor(rho, device=device, dtype=torch.float64)

    args = (S0, K, V0, rf, lambd, tau, kappa, theta, sigma, rho)


    gl_x = torch.as_tensor(_gl_nodes_np, device=device, dtype=torch.float64)
    gl_w = torch.as_tensor(_gl_weights_np, device=device, dtype=torch.float64)
    gl_w_adj = gl_w * torch.exp(gl_x)

    phi_c = gl_x.to(torch.complex64)
    phi_grid = phi_c.unsqueeze(0).expand(batch_n, -1)

    args_batch = []
    for i in args:
        if i.ndim == 1:
            args_batch.append(i.unsqueeze(-1))
        else:
            args_batch.append(i.reshape(1,1).expand(batch_n, -1))

    vals = integrand(phi_grid, *args_batch)
    integral = torch.sum(vals * gl_w_adj.unsqueeze(0), dim=1)
    real_integral = torch.real(integral)
    call = (S0 - K * torch.exp(-rf * tau)) / 2.0 + real_integral / torch.pi

    return torch.maximum(call, torch.tensor(0.0, device=device))




def put_heston_gpu(call_price, S0, K, rf, tau):
    device = torch.device('cuda')
    S0 = torch.as_tensor(S0, device=device, dtype=torch.float64)
    K = torch.as_tensor(K, device=device, dtype=torch.float64)
    rf = torch.as_tensor(rf, device=device, dtype=torch.float64)
    tau = torch.as_tensor(tau, device=device, dtype=torch.float64)
    put = torch.exp(-rf*tau)*K + call_price - S0

    return torch.maximum(put, torch.tensor(0.0, device=device))



def heston_avg_var(V0, kappa, theta, tau):
    
    device = (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
    V0    = torch.as_tensor(V0.to_numpy(),    device=device, dtype=torch.float64)
    kappa = torch.as_tensor(kappa.to_numpy(), device=device, dtype=torch.float64)
    theta = torch.as_tensor(theta.to_numpy(), device=device, dtype=torch.float64)
    tau   = torch.as_tensor(tau.to_numpy(),   device=device, dtype=torch.float64)

    safe_kappa = kappa.clamp(min=1e-8)
    avg_var = theta + (V0 - theta) * (1.0 - torch.exp(-safe_kappa * tau)) / (safe_kappa * tau)
    return torch.sqrt(avg_var.clamp(min=0.0))






def SABR_vol(alpha,beta,rho,nu,F,K,tau):
    device = torch.device('cuda')
    eps = 1e-8
    alpha = torch.as_tensor(alpha, device=device, dtype=torch.float64)
    beta = torch.as_tensor(beta, device=device, dtype=torch.float64)
    rho = torch.as_tensor(rho, device=device, dtype=torch.float64)
    nu = torch.as_tensor(nu, device=device, dtype=torch.float64)
    F = torch.as_tensor(F, device=device, dtype=torch.float64)
    K = torch.as_tensor(K, device=device, dtype=torch.float64)
    tau = torch.as_tensor(tau, device=device, dtype=torch.float64)
    F = torch.clamp(torch.as_tensor(F, device=device, dtype=torch.float64), min=eps)
    K = torch.clamp(torch.as_tensor(K, device=device, dtype=torch.float64), min=eps)

    condition = torch.isclose(F, K, rtol=1e-5)

    V = (F*K)**((1-beta)/2.)
    logFK = torch.log(torch.clamp(F/K, min=eps))
    A = 1 + ( ((1-beta)**2*alpha**2)/(24.*(V**2)) + (alpha*beta*nu*rho)/(4.*V) + ((nu**2)*(2-3*(rho**2))/24.) ) * tau
    
    VOL_eq = (alpha/V)*A

    z = (nu/alpha)*V*logFK
    x = torch.log((torch.sqrt(1-2*rho*z+z**2) + z - rho)/(1-rho))
    B = 1 + (1/24.)*(((1-beta)*logFK)**2) + (1/1920.)*(((1-beta)*logFK)**4)
    
    VOL_ineq = (nu*logFK*A)/(x*B)

    VOL = torch.where(condition, VOL_eq, VOL_ineq)

    return (VOL)


def BS_pricer(S0, K, rf, impl_vol, tau, corp):
    
    batch_n = S0.shape[0]

    device = torch.device('cuda')
    S0 = torch.as_tensor(S0, device=device, dtype=torch.float64)
    K = torch.as_tensor(K, device=device, dtype=torch.float64)
    rf = torch.as_tensor(rf, device=device, dtype=torch.float64)
    impl_vol = torch.as_tensor(impl_vol, device=device, dtype=torch.float64)
    tau = torch.as_tensor(tau, device=device, dtype=torch.float64)
    corp = torch.as_tensor(corp, device=device, dtype=torch.float64)
    
    if not torch.isfinite(impl_vol).all():
        print("impl_vol:", impl_vol)
        print(impl_vol[~torch.isfinite(impl_vol)])
    d1 = 1/(impl_vol*torch.sqrt(tau))*(torch.log(S0/K) + (rf + (impl_vol**2)/2)*tau)
    d2 = d1 - impl_vol*torch.sqrt(tau)
    BS_price = corp*torch.distributions.Normal(0, 1).cdf(corp*d1)*S0 - corp*torch.distributions.Normal(0, 1).cdf(corp*d2)*K*torch.exp(-rf*tau)
    return(BS_price)


def BS_impl_vol(S0, K, rf, P, tau, corp):
    def obj_funct(impl_vol_guess):
        return((P - BS_pricer(S0, K, rf, impl_vol_guess, tau, corp))**2)
    
    impl_vol_guess = torch.tensor([0.03], requires_grad=True, device='cuda')


    optimizer = optim.Adam([impl_vol_guess], lr=0.01)

    num_iterations = 1000
    for i in range(num_iterations):
        optimizer.zero_grad()
        loss = obj_funct(impl_vol_guess)
        loss.backward()
        optimizer.step()
    
    for i in range(num_iterations):
        y = obj_funct(impl_vol_guess)
        grad = torch.autograd.grad(y, impl_vol_guess, create_graph=True)[0]
        hess = torch.autograd.grad(grad, impl_vol_guess, create_graph=True)[0]
        impl_vol_guess = impl_vol_guess - grad/hess


    
    impl_var_optimized = impl_vol_guess.item()
    impl_vol_optimized = torch.sqrt(impl_vol_guess).item()

    print(f"Optimized Variance: {impl_var_optimized}")
    print(f"Instantaneous Volatility: {impl_vol_optimized}")

    return(impl_vol_optimized)

In [ ]:
# GPU Implied Vol estimator (Newton Rhapson method) with batching and parallelization

def bs_price_torch(S, K, T, r, cp, sigma):
    eps = 1e-12
    sigma = torch.clamp(sigma, min=eps)
    T = torch.clamp(T, min=eps)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * torch.sqrt(T))
    d2 = d1 - sigma * torch.sqrt(T)

    normal = torch.distributions.Normal(0.0, 1.0)
    return cp * (S * normal.cdf(cp * d1) - K * torch.exp(-r * T) * normal.cdf(cp * d2))

def bs_vega_torch(S, K, T, r, sigma):
    eps = 1e-12
    sigma = torch.clamp(sigma, min=eps)
    T = torch.clamp(T, min=eps)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * torch.sqrt(T))
    normal = torch.distributions.Normal(0.0, 1.0)
    return S * normal.log_prob(d1).exp() * torch.sqrt(T)


def implied_vol_newton_gpu_batched(
    dfs: list[pd.DataFrame],
    price_cols: list[str],
    batch_size: int = 10_000,
    initial_guess: float = 0.2,
    tol: float = 1e-10,
    max_iter: int = 100,
) -> list[torch.Tensor]:
    
    dtype  = torch.float64
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n      = len(dfs[0])
    N      = len(dfs)

    S_np     = [df["stock_price"].values  for df in dfs]
    K_np     = [df["strike_price"].values for df in dfs]
    T_np     = [df["time_to_mat"].values  for df in dfs]
    r_np     = [df["rf"].values           for df in dfs]
    cp_np    = [df["cp_flag_one"].values  for df in dfs]
    price_np = [df[col].values for df, col in zip(dfs, price_cols)]

    results = [torch.empty(n, dtype=dtype) for _ in range(N)]

    for batch_start in range(0, n, batch_size):
        batch_end = min(batch_start + batch_size, n)
        sl = slice(batch_start, batch_end)

        def to_gpu(arr):
            return torch.as_tensor(arr[sl], dtype=dtype, device=device)

        S  = torch.stack([to_gpu(s)  for s  in S_np],  dim=0)
        K  = torch.stack([to_gpu(k)  for k  in K_np],  dim=0)
        T  = torch.stack([to_gpu(t)  for t  in T_np],  dim=0)
        r  = torch.stack([to_gpu(rv) for rv in r_np],  dim=0)
        cp = torch.stack([to_gpu(c)  for c  in cp_np], dim=0)

        prices = torch.stack(
            [torch.as_tensor(p[sl], dtype=dtype, device=device) for p in price_np],
            dim=0,
        )

        sigma = torch.full((N, batch_end - batch_start), initial_guess,
                           dtype=dtype, device=device)

        for _ in range(max_iter):
            price_est = bs_price_torch(S, K, T, r, cp, sigma)
            vega      = bs_vega_torch(S, K, T, r, sigma)

            diff      = price_est - prices
            converged = torch.abs(diff) < tol

            if torch.all(converged):
                break

            vega  = torch.clamp(vega, min=1e-12)
            sigma = sigma - diff / vega
            sigma = torch.clamp(sigma, min=1e-8, max=5.0)

        for i in range(N):
            results[i][sl] = sigma[i].cpu()

        del S, K, T, r, cp, prices, sigma, price_est, vega, diff, converged
        torch.cuda.empty_cache()

    return results


def bs_price_from_iv_gpu_batched(
    dfs: list[pd.DataFrame],
    iv_cols: list[str],
    batch_size: int = 10_000,
) -> list[torch.Tensor]:
    
    dtype  = torch.float64
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n      = len(dfs[0])
    N      = len(dfs)

    S_np  = [df["stock_price"].values  for df in dfs]
    K_np  = [df["strike_price"].values for df in dfs]
    T_np  = [df["time_to_mat"].values  for df in dfs]
    r_np  = [df["rf"].values           for df in dfs]
    cp_np = [df["cp_flag_one"].values  for df in dfs]
    iv_np = [df[col].values for df, col in zip(dfs, iv_cols)]

    results = [torch.empty(n, dtype=dtype) for _ in range(N)]

    for batch_start in range(0, n, batch_size):
        batch_end = min(batch_start + batch_size, n)
        sl = slice(batch_start, batch_end)

        def to_gpu(arr):
            return torch.as_tensor(arr[sl], dtype=dtype, device=device)

        S  = torch.stack([to_gpu(s)  for s  in S_np],  dim=0)
        K  = torch.stack([to_gpu(k)  for k  in K_np],  dim=0)
        T  = torch.stack([to_gpu(t)  for t  in T_np],  dim=0)
        r  = torch.stack([to_gpu(rv) for rv in r_np],  dim=0)
        cp = torch.stack([to_gpu(c)  for c  in cp_np], dim=0)

        sigmas = torch.stack(
            [torch.as_tensor(iv[sl], dtype=dtype, device=device) for iv in iv_np],
            dim=0,
        )

        prices = bs_price_torch(S, K, T, r, cp, sigmas)

        for i in range(N):
            results[i][sl] = prices[i].cpu()

        del S, K, T, r, cp, sigmas, prices
        torch.cuda.empty_cache()

    return results

In [ ]:
# 2 Step model function definition both for Heston and Sabr

def two_step_stochvol_ml(
    data,
    model_type: str,
    ml_type:    str,
    train_size: float = 0.7,
    batch_size: int   = 2048,
    n_epochs:   int   = 1000,
    max_iter:   int   = 200,
    patience:   int   = 10,
    params:     dict  = None,
    seed:       int   = 42,
    verbose:    bool  = True,
):
    
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    _p = params or {}

    if model_type == "heston":
        param_names  = ["V0", "theta", "sigma", "rho"]
        param_bounds = [(1e-3, 0.30), (1e-3, 0.30), (1e-2, 1.0), (-0.999, 0.0)]
        param_x0     = [0.10, 0.05, 0.30, -0.50]
        mkt_cols     = ["stock_price", "strike_price", "time_to_mat", "rf", "VRP", "cp_flag_one"]
        target_col   = "price"
    else:
        param_names  = ["alphaS", "rhoS", "nuS"]
        param_bounds = [(1e-3, 10.0), (-0.999, -0.001), (1e-3, 10.0)]
        param_x0     = [0.10, -0.10, 0.05]
        mkt_cols     = ["forward_price", "stock_price", "strike_price", "time_to_mat", "rf", "cp_flag_one"]
        target_col   = "price"

    n_params = len(param_names)
    lbs = torch.tensor([b[0] for b in param_bounds], dtype=torch.float64, device=device)
    ubs = torch.tensor([b[1] for b in param_bounds], dtype=torch.float64, device=device)

    no_scale_cols = {"cp_flag_one"}

    is_rho = ["rho" in name.lower() for name in param_names]

    def _inverse_sp(x_phys, low, high):
        scaled = (x_phys - low) / (high - low + 1e-12)
        sp = scaled / (1 - scaled + 1e-12)
        return np.log(np.expm1(sp))

    def _inverse_tanh(x_phys, low, high):
        center = 0.5 * (low + high)
        half_range = 0.5 * (high - low)
        norm = (x_phys - center) / (half_range + 1e-12)
        return np.arctanh(np.clip(norm, -0.999, 0.999))

    def _forward_map(raw):
        p = torch.empty_like(raw)
        for j in range(n_params):
            if is_rho[j]:
                t = torch.tanh(raw[j])
                p[j] = lbs[j] + (ubs[j] - lbs[j]) * (t + 1.0) / 2.0
            else:
                sp = torch.nn.functional.softplus(raw[j])
                p[j] = lbs[j] + (ubs[j] - lbs[j]) * sp / (1.0 + sp)
        return p

    def compute_sv_batch(df, params_np, return_ivol=False):
        N = len(df); out = np.zeros(N, dtype=np.float64)
        out_ivol = np.zeros(N, dtype=np.float64) if (model_type == "sabr" and return_ivol) else None
        for s in range(0, N, batch_size):
            e = min(s + batch_size, N)
            def t(arr): return torch.tensor(arr, dtype=torch.float64, device=device)
            if model_type == "heston":
                S0  = t(df["stock_price"].values[s:e]); K  = t(df["strike_price"].values[s:e])
                tau = t(df["time_to_mat"].values[s:e]); rf = t(df["rf"].values[s:e])
                lam = t(df["VRP"].values[s:e]);         cp = t(df["cp_flag_one"].values[s:e])
                V0  = t(params_np[s:e,0]); kap = torch.tensor(2.0, dtype=torch.float64, device=device)
                tht = t(params_np[s:e,1]); sig = t(params_np[s:e,2]); rho = t(params_np[s:e,3])
                call  = heston_price_gpu(S0, K, V0, rf, lam, tau, kap, tht, sig, rho)
                price = torch.where(cp > 0, call, put_heston_gpu(call, S0, K, rf, tau))
                out[s:e] = price.cpu().numpy()
            else:
                F   = t(df["forward_price"].values[s:e]); K  = t(df["strike_price"].values[s:e])
                tau = t(df["time_to_mat"].values[s:e])
                S0  = t(df["stock_price"].values[s:e]); rf = t(df["rf"].values[s:e])
                cp  = t(df["cp_flag_one"].values[s:e])
                alp = t(params_np[s:e,0]); bet = torch.ones(e-s,dtype=torch.float64,device=device)
                rho = t(params_np[s:e,1]); nu  = t(params_np[s:e,2])
                sabr_ivol    = SABR_vol(alp, bet, rho, nu, F, K, tau)
                out[s:e] = BS_pricer(S0, K, rf, sabr_ivol, tau, cp).cpu().numpy()
                if return_ivol:
                    out_ivol[s:e] = sabr_ivol.cpu().numpy()
        if return_ivol:
            return out, out_ivol
        return out

    def compute_sv_global(df, global_params, return_ivol=False):
        N = len(df)
        params_np = np.tile(np.array(global_params, dtype=np.float64), (N, 1))
        return compute_sv_batch(df, params_np, return_ivol=return_ivol)

    data = data.copy().reset_index(drop=True)
    if "cp_flag_one" not in data.columns:
        data["cp_flag_one"] = data["cp_flag"].apply(lambda x: 1.0 if x == "C" else -1.0)

    dates  = data["date"].sort_values().unique()
    cutoff = dates[int(len(dates) * train_size)]
    train_mask = data["date"] < cutoff
    train_all = data[train_mask].copy().reset_index(drop=True)
    test_data = data[~train_mask].copy().reset_index(drop=True)

    split = len(train_all) // 2
    set_a = train_all.iloc[:split].copy().reset_index(drop=True)
    set_b = train_all.iloc[split:].copy().reset_index(drop=True)

    if verbose:
        print(f"Train: {len(train_all):,} (Set A: {len(set_a):,} | Set B: {len(set_b):,})  |  Test: {len(test_data):,}")

    lhs_unit   = LatinHypercube(d=n_params, seed=seed).random(n=len(set_a))
    params_lhs = np.array([lb + (ub-lb)*lhs_unit[:,j]
                            for j,(lb,ub) in enumerate(param_bounds)]).T
    if verbose: print("Computing stoch-vol targets for Set A (LHS)...")
    sv_a  = compute_sv_batch(set_a, params_lhs)
    valid = np.isfinite(sv_a) & (sv_a > 0)
    if verbose and (~valid).sum() > 0:
        print(f"  Dropping {(~valid).sum()} non-finite / non-positive rows")
    set_a_v  = set_a[valid].reset_index(drop=True)
    params_v = params_lhs[valid]
    sv_v     = sv_a[valid]

    all_feat_cols = mkt_cols + [f"_p{j}" for j in range(n_params)]
    X_mkt = set_a_v[mkt_cols].values.astype(np.float32)
    X_all = np.concatenate([X_mkt, params_v.astype(np.float32)], axis=1)
    y_all = np.log(sv_v).astype(np.float32)

    n_s1_tr = int(len(y_all) * 0.85)
    X_tr, X_te_s1 = X_all[:n_s1_tr], X_all[n_s1_tr:]
    y_tr, y_te_s1 = y_all[:n_s1_tr], y_all[n_s1_tr:]
    y_te_s1_raw   = sv_v[n_s1_tr:]
    n_feat = X_all.shape[1]; n_mkt = len(mkt_cols)

    scale_mask = np.array([c not in no_scale_cols for c in mkt_cols] + [True]*n_params)

    if ml_type in ("adam", "lbfgs"):
        scaler_x = StandardScaler()
        scaler_x.fit(X_tr[:, scale_mask])
        def _scale_x(X):
            Xs = X.copy()
            Xs[:, scale_mask] = scaler_x.transform(X[:, scale_mask])
            return Xs.astype(np.float32)
        X_tr_sc    = _scale_x(X_tr)
        X_te_s1_sc = _scale_x(X_te_s1)
    else:
        X_tr_sc    = X_tr
        X_te_s1_sc = X_te_s1

    n_val  = max(1, int(n_s1_tr * 0.15))
    n_tr2  = n_s1_tr - n_val
    X_tr2_sc, X_val_sc = X_tr_sc[:n_tr2], X_tr_sc[n_tr2:]
    y_tr2, y_val       = y_tr[:n_tr2],    y_tr[n_tr2:]

    if ml_type in ("adam", "lbfgs"):
        mean_x_full = np.zeros(n_feat, dtype=np.float64)
        std_x_full  = np.ones(n_feat,  dtype=np.float64)
        mean_x_full[scale_mask] = scaler_x.mean_
        std_x_full[scale_mask]  = scaler_x.scale_
        mean_x_t = torch.tensor(mean_x_full, dtype=torch.float64, device=device)
        std_x_t  = torch.tensor(std_x_full,  dtype=torch.float64, device=device)
        mean_mkt_t    = mean_x_t[:n_mkt];  std_mkt_t    = std_x_t[:n_mkt]
        mean_params_t = mean_x_t[n_mkt:];  std_params_t = std_x_t[n_mkt:]

    if verbose:
        print(f"Step 1 – {ml_type} | {n_tr2:,} train  {n_val:,} val  {len(y_te_s1):,} test"
              f"  {n_feat} features  (log Y)")

    if ml_type in ("adam", "lbfgs"):
        n_layers = _p.get('n_layers', 3)
        n_units  = _p.get('n_units',  64)

        class _MLP(nn.Module):
            def __init__(self, n_in):
                super().__init__()
                layers = [nn.Linear(n_in, n_units), nn.GELU()]
                for _ in range(n_layers - 1):
                    layers += [nn.Linear(n_units, n_units), nn.GELU()]
                layers.append(nn.Linear(n_units, 1))
                self.net = nn.Sequential(*layers)
            def forward(self, x): return self.net(x).squeeze(-1)

        ann    = _MLP(n_feat).to(device).double()
        loss_f = nn.MSELoss()
        Xtr2_t = torch.tensor(X_tr2_sc, dtype=torch.float64, device=device)
        ytr2_t = torch.tensor(y_tr2,    dtype=torch.float64, device=device)
        Xval_t = torch.tensor(X_val_sc, dtype=torch.float64, device=device)
        yval_t = torch.tensor(y_val,    dtype=torch.float64, device=device)

        best_val   = float('inf')
        best_state = {k: v.clone() for k, v in ann.state_dict().items()}
        no_improve = 0

        if ml_type == "adam":
            opt = optim.Adam(ann.parameters(), lr=1e-3)
            for ep in range(n_epochs):
                ann.train()
                perm = torch.randperm(len(Xtr2_t), device=device)
                for i in range(0, len(Xtr2_t), batch_size):
                    xb = Xtr2_t[perm[i:i+batch_size]]; yb = ytr2_t[perm[i:i+batch_size]]
                    opt.zero_grad(); loss_f(ann(xb), yb).backward(); opt.step()
                ann.eval()
                with torch.no_grad():
                    val_loss = loss_f(ann(Xval_t), yval_t).item()
                if val_loss < best_val:
                    best_val = val_loss
                    best_state = {k: v.clone() for k, v in ann.state_dict().items()}
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        if verbose: print(f"  Early stop epoch {ep+1} | best val: {best_val:.6f}")
                        break
                if verbose and (ep + 1) % max(1, n_epochs // 5) == 0:
                    print(f"  Epoch {ep+1}/{n_epochs} | val MSE: {val_loss:.6f}")

        else:
            opt = optim.LBFGS(ann.parameters(), lr=1.0, max_iter=20, history_size=10)
            for ep in range(n_epochs//10):
                ann.train()
                def _closure():
                    opt.zero_grad()
                    l = loss_f(ann(Xtr2_t), ytr2_t)
                    l.backward()
                    return l
                opt.step(_closure)
                ann.eval()
                with torch.no_grad():
                    val_loss = loss_f(ann(Xval_t), yval_t).item()
                if val_loss < best_val:
                    best_val = val_loss
                    best_state = {k: v.clone() for k, v in ann.state_dict().items()}
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        if verbose: print(f"  Early stop epoch {ep+1} | best val: {best_val:.6f}")
                        break
                if verbose and (ep + 1) % max(1, n_epochs // 5) == 0:
                    print(f"  Epoch {ep+1}/{n_epochs} | val MSE: {val_loss:.6f}")

        ann.load_state_dict(best_state)

    else:
        model_xgb = xgb_lib.XGBRegressor(
            gamma=0.01, learning_rate=0.1, max_depth=_p.get('max_depth', 8),
            n_estimators=_p.get('n_estimators', 500), subsample=1.0, reg_lambda=10,
            n_jobs=-1, random_state=seed,
            early_stopping_rounds=patience,
            device=device.type,
        )
        model_xgb.fit(X_tr2_sc, y_tr2,
                      eval_set=[(X_val_sc, y_val)],
                      verbose=50 if verbose else False)

    if ml_type in ("adam", "lbfgs"):
        ann.eval()
        Xte_s1_t = torch.tensor(X_te_s1_sc, dtype=torch.float64, device=device)
        with torch.no_grad():
            y_pred1_log = ann(Xte_s1_t).cpu().numpy()
    else:
        y_pred1_log = model_xgb.predict(X_te_s1_sc)

    y_pred1 = np.exp(y_pred1_log)
    mse1  = mean_squared_error(y_te_s1_raw, y_pred1)
    rmse1 = np.sqrt(mse1)
    mape1 = mean_absolute_percentage_error(y_te_s1_raw, y_pred1)
    r2_1  = r2_score(y_te_s1_raw, y_pred1)
    print(f"\n=== Step 1 | {model_type.upper()} | {ml_type} ===")
    print(f"  MSE:  {mse1:.6f}  RMSE: {rmse1:.6f}")
    print(f"  MAPE: {mape1:.6f}  R²:   {r2_1:.6f}")

    raw_init = np.array([
        _inverse_tanh(x0, lb, ub) if is_rho[j] else _inverse_sp(x0, lb, ub)
        for j, (x0, (lb, ub)) in enumerate(zip(param_x0, param_bounds))
    ], dtype=np.float64)

    if verbose: print(f"\nStep 2 – calibrating global params on Set B ({len(set_b):,} rows)...")

    raw_global = torch.tensor(raw_init, dtype=torch.float64, device=device, requires_grad=True)

    X_b_mkt = set_b[mkt_cols].values.astype(np.float32)
    y_b_log = np.log(np.clip(set_b[target_col].values.astype(np.float64), 1e-8, None))

    if ml_type == "xgboost":
        def _xgb_pred_global(p_bnd_np):
            params_tile = np.tile(p_bnd_np.astype(np.float32), (len(X_b_mkt), 1))
            x_in = np.concatenate([X_b_mkt, params_tile], axis=1)
            return model_xgb.predict(x_in).astype(np.float64)

        opt2 = optim.LBFGS([raw_global], lr=1.0, max_iter=max_iter,
                           history_size=10, line_search_fn="strong_wolfe",
                           tolerance_grad=1e-5, tolerance_change=1e-7)
        eps = 1e-4
        _xgb_iter = [0]

        def _xgb_closure_global():
            _xgb_iter[0] += 1
            with torch.no_grad():
                p_bnd = _forward_map(raw_global).cpu().numpy()
            base_pred = _xgb_pred_global(p_bnd)
            base_loss = ((base_pred - y_b_log)**2).mean()
            grad_raw = np.zeros(n_params, dtype=np.float64)
            raw_np = raw_global.detach().cpu().numpy()
            for j in range(n_params):
                raw_p = raw_np.copy(); raw_p[j] += eps
                raw_m = raw_np.copy(); raw_m[j] -= eps
                with torch.no_grad():
                    p_p = _forward_map(torch.tensor(raw_p, dtype=torch.float64, device=device)).cpu().numpy()
                    p_m = _forward_map(torch.tensor(raw_m, dtype=torch.float64, device=device)).cpu().numpy()
                loss_p = ((_xgb_pred_global(p_p) - y_b_log)**2).mean()
                loss_m = ((_xgb_pred_global(p_m) - y_b_log)**2).mean()
                grad_raw[j] = (loss_p - loss_m) / (2 * eps)
            raw_global.grad = torch.tensor(grad_raw, dtype=torch.float64, device=device)
            if verbose and _xgb_iter[0] % 20 == 0:
                print(f"  LBFGS iter {_xgb_iter[0]} | loss: {base_loss:.6f}")
            return torch.tensor(base_loss, dtype=torch.float64, device=device)

        opt2.step(_xgb_closure_global)

    else:
        ann.eval()
        for p in ann.parameters(): p.requires_grad_(False)

        Xb_t_sc = (torch.tensor(X_b_mkt, dtype=torch.float64, device=device)
                   - mean_mkt_t) / std_mkt_t
        yb_t    = torch.tensor(y_b_log, dtype=torch.float64, device=device)

        opt2 = optim.LBFGS([raw_global], lr=1.0, max_iter=max_iter,
                           history_size=10, line_search_fn="strong_wolfe")
        _ann_iter = [0]

        def _cl():
            _ann_iter[0] += 1
            opt2.zero_grad()
            p_bnd    = _forward_map(raw_global)
            p_bnd_sc = (p_bnd - mean_params_t) / std_params_t
            p_bnd_sc_exp = p_bnd_sc.unsqueeze(0).expand(len(Xb_t_sc), -1)
            l = ((ann(torch.cat([Xb_t_sc, p_bnd_sc_exp], dim=1)) - yb_t)**2).mean()
            if verbose and _ann_iter[0] % 20 == 0:
                print(f"  LBFGS iter {_ann_iter[0]} | loss: {l.item():.6f}")
            l.backward()
            return l

        opt2.step(_cl)

    with torch.no_grad():
        cal_params_global = _forward_map(raw_global).cpu().numpy()

    if verbose:
        param_str = ", ".join(f"{n}={v:.6f}" for n,v in zip(param_names, cal_params_global))
        print(f"  Calibrated params: {param_str}")


    if verbose: print("\nEvaluating ML surrogate on full dataset...")
    X_full_mkt = data[mkt_cols].values.astype(np.float32)
    params_tile_full = np.tile(cal_params_global.astype(np.float32), (len(data), 1))
    X_full_in = np.concatenate([X_full_mkt, params_tile_full], axis=1)

    if ml_type in ("adam", "lbfgs"):
        X_full_sc = _scale_x(X_full_in)
        ann.eval()
        ml_pred_log = []
        with torch.no_grad():
            for s in range(0, len(X_full_sc), batch_size):
                e = min(s + batch_size, len(X_full_sc))
                xb = torch.tensor(X_full_sc[s:e], dtype=torch.float64, device=device)
                ml_pred_log.append(ann(xb).cpu().numpy())
        ml_pred_log = np.concatenate(ml_pred_log)
    else:
        ml_pred_log = model_xgb.predict(X_full_in)

    ml_pred_full = np.exp(ml_pred_log)

    if verbose: print("Evaluating actual stoch-vol pricer on full dataset...")
    _sv_full_res = compute_sv_global(data, cal_params_global,
                                     return_ivol=(model_type == "sabr"))
    if model_type == "sabr":
        sv_price_full, sv_ivol_full = _sv_full_res
    else:
        sv_price_full = _sv_full_res
        sv_ivol_full = None

    test_idx = np.where(~train_mask.values)[0]
    y_test_mkt = data.loc[test_idx, target_col].values.astype(np.float64)

    ml_pred_test = ml_pred_full[test_idx]
    ok_ml = np.isfinite(ml_pred_test) & np.isfinite(y_test_mkt)
    mse_ml  = mean_squared_error(y_test_mkt[ok_ml], ml_pred_test[ok_ml])
    rmse_ml = np.sqrt(mse_ml)
    mape_ml = mean_absolute_percentage_error(y_test_mkt[ok_ml], ml_pred_test[ok_ml])
    r2_ml   = r2_score(y_test_mkt[ok_ml], ml_pred_test[ok_ml])

    print(f"\n=== ML Surrogate (test set) | {model_type.upper()} | {ml_type} ===")
    print(f"  MSE:  {mse_ml:.6f}  RMSE: {rmse_ml:.6f}")
    print(f"  MAPE: {mape_ml:.6f}  R²:   {r2_ml:.6f}")

    sv_pred_test = sv_price_full[test_idx]
    ok_sv = np.isfinite(sv_pred_test) & np.isfinite(y_test_mkt)
    mse_sv  = mean_squared_error(y_test_mkt[ok_sv], sv_pred_test[ok_sv])
    rmse_sv = np.sqrt(mse_sv)
    mape_sv = mean_absolute_percentage_error(y_test_mkt[ok_sv], sv_pred_test[ok_sv])
    r2_sv   = r2_score(y_test_mkt[ok_sv], sv_pred_test[ok_sv])

    print(f"\n=== Stoch-Vol Pricer (test set) | {model_type.upper()} | {ml_type} ===")
    print(f"  MSE:  {mse_sv:.6f}  RMSE: {rmse_sv:.6f}")
    print(f"  MAPE: {mape_sv:.6f}  R²:   {r2_sv:.6f}")

    data_out = data.copy()
    data_out["out_of_sample"] = (~train_mask).astype(int).values

    for j, pname in enumerate(param_names):
        data_out[pname] = cal_params_global[j]

    if model_type == "heston":
        data_out["heston_price_ml"] = ml_pred_full
    else:
        data_out["sabr_price_ml"] = ml_pred_full

    if model_type == "heston":
        data_out["heston_price"] = sv_price_full
        data_out["kappa"] = 2.0
        inst_vol = heston_avg_var(
            data_out["V0"], data_out["kappa"], data_out["theta"],
            data_out["time_to_mat"]
        ).cpu().numpy()
        data_out["inst_vol_heston"] = inst_vol
        iv_sv, iv_ml = implied_vol_newton_gpu_batched(
            [data_out, data_out],
            ["heston_price", "heston_price_ml"],
        )
        data_out["heston_volatility"] = iv_sv.numpy()
        data_out["heston_volatility_ml"] = iv_ml.numpy()
    else:
        data_out["sabr_price"] = sv_price_full
        data_out["sabr_volatility"] = sv_ivol_full

    data_out["ml_mse_optim"]      = mse_ml
    data_out["ml_rmse_optim"]     = rmse_ml
    data_out["ml_mape_optim"]     = mape_ml
    data_out["ml_rsquared_optim"] = r2_ml
    data_out["mse_optim"]         = mse_sv
    data_out["rmse_optim"]        = rmse_sv
    data_out["mape_optim"]        = mape_sv
    data_out["rsquared_optim"]    = r2_sv

    metrics = dict(
        step1_mse=mse1, step1_rmse=rmse1, step1_mape=mape1, step1_r2=r2_1,
        ml_mse=mse_ml, ml_rmse=rmse_ml, ml_mape=mape_ml, ml_r2=r2_ml,
        sv_mse=mse_sv, sv_rmse=rmse_sv, sv_mape=mape_sv, sv_r2=r2_sv,
    )
    return data_out, metrics

In [ ]:
# Load Optuna optimised hyperparameters

_OPTUNA_PATH = r'C:\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\optuna_best_params_2003.json'
if _os.path.exists(_OPTUNA_PATH):
    with open(_OPTUNA_PATH) as _f:
        _optuna = _json.load(_f)
    print(f"Loaded {len(_optuna)} param sets from {_OPTUNA_PATH}")
else:
    _optuna = {}
    print(f"No Optuna params found at {_OPTUNA_PATH} — using defaults")

def _get_params(key):
    raw = _optuna.get(key, {})
    if not raw:
        return None
    out = {}
    if 'n_layers' in raw: out['n_layers'] = raw['n_layers']
    if 'n_units'  in raw: out['n_units']  = raw['n_units']
    if 'max_depth'     in raw: out['max_depth']     = raw['max_depth']
    if 'n_estimators'  in raw: out['n_estimators']  = raw['n_estimators']
    if 'learning_rate' in raw: out['learning_rate']  = raw['learning_rate']
    if 'subsample'     in raw: out['subsample']      = raw['subsample']
    if 'reg_lambda'    in raw: out['reg_lambda']     = raw['reg_lambda']
    if 'gamma'         in raw: out['gamma']          = raw['gamma']
    return out if out else None

for k in sorted(_optuna.keys()):
    print(f"  {k}: {_optuna[k]}")

Loaded 8 param sets from C:\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\optuna_best_params_2003.json
  one_step_lbfgs_heston: {'n_layers': 3, 'n_units': 128, 'max_iter': 50, 'history_size': 11}
  one_step_lbfgs_sabr: {'n_layers': 3, 'n_units': 64, 'max_iter': 46, 'history_size': 13}
  one_step_xgb_heston: {'learning_rate': 0.15481298363980872, 'max_depth': 7, 'n_estimators': 459, 'subsample': 0.9508777032363638, 'reg_lambda': 1.6632830286071554, 'gamma': 0.42402836938973243}
  one_step_xgb_sabr: {'learning_rate': 0.020464381773418736, 'max_depth': 12, 'n_estimators': 291, 'subsample': 0.9029270751913427, 'reg_lambda': 2.585993121087356, 'gamma': 4.54577045901569}
  two_step_lbfgs_heston: {'n_layers': 3, 'n_units': 128, 'max_iter': 40, 'history_size': 15}
  two_step_lbfgs_sabr: {'n_layers': 4, 'n_units': 64, 'max_iter': 36, 'history_size': 9}
  two_step_xgb_heston: {'learning_rate': 0.03510733639050293, 'max_depth': 

In [ ]:
# XGBoost – Heston
set_seed(42)
out_xgb_heston, metrics_xgb_heston = two_step_stochvol_ml(
    data, model_type="heston", ml_type="xgboost",
    train_size=0.7, batch_size=2048, max_iter=200,
    params=_get_params("two_step_xgb_heston"),
    seed=42, verbose=True,
)

# XGBoost – Sabr
set_seed(42)
out_xgb_sabr, metrics_xgb_sabr = two_step_stochvol_ml(
    data, model_type="sabr", ml_type="xgboost",
    train_size=0.7, batch_size=2048, max_iter=200,
    params=_get_params("two_step_xgb_sabr"),
    seed=42, verbose=True,
)

Train: 117,065 (Set A: 58,532 | Set B: 58,533)  |  Test: 46,777
Computing stoch-vol targets for Set A (LHS)...
  Dropping 832 non-finite / non-positive rows
Step 1 – xgboost | 41,689 train  7,356 val  8,655 test  10 features  (log Y)
[0]	validation_0-rmse:1.94888
[50]	validation_0-rmse:0.37690
[100]	validation_0-rmse:0.35549
[150]	validation_0-rmse:0.35156
[168]	validation_0-rmse:0.35141

=== Step 1 | HESTON | xgboost ===
  MSE:  3927.480349  RMSE: 62.669613
  MAPE: 0.840012  R²:   0.675757

Step 2 – calibrating global params on Set B (58,533 rows)...
  Calibrated params: V0=0.100000, theta=0.050000, sigma=0.300000, rho=-0.500000

Evaluating ML surrogate on full dataset...
Evaluating actual stoch-vol pricer on full dataset...

=== ML Surrogate (test set) | HESTON | xgboost ===
  MSE:  9601.582900  RMSE: 97.987667
  MAPE: 2.841755  R²:   0.034913

=== Stoch-Vol Pricer (test set) | HESTON | xgboost ===
  MSE:  210.317576  RMSE: 14.502330
  MAPE: 1.024460  R²:   0.978860
Train: 117,065 (S

In [ ]:
# ANN LBFGS – Heston
set_seed(42)
out_lbfgs_heston, metrics_lbfgs_heston = two_step_stochvol_ml(
    data, model_type="heston", ml_type="lbfgs",
    train_size=0.7, batch_size=2048, n_epochs=100, max_iter=200,
    params=_get_params("two_step_lbfgs_heston"),
    seed=42, verbose=True,
)

# ANN LBFGS – Sabr
set_seed(42)
out_lbfgs_sabr, metrics_lbfgs_sabr = two_step_stochvol_ml(
    data, model_type="sabr", ml_type="lbfgs",
    train_size=0.7, batch_size=2048, n_epochs=100, max_iter=200,
    params=_get_params("two_step_lbfgs_sabr"),
    seed=42, verbose=True,
)

Train: 117,065 (Set A: 58,532 | Set B: 58,533)  |  Test: 46,777
Computing stoch-vol targets for Set A (LHS)...
  Dropping 832 non-finite / non-positive rows
Step 1 – lbfgs | 41,689 train  7,356 val  8,655 test  10 features  (log Y)

=== Step 1 | HESTON | lbfgs ===
  MSE:  1538.517117  RMSE: 39.223936
  MAPE: 0.355152  R²:   0.872984

Step 2 – calibrating global params on Set B (58,533 rows)...
  LBFGS iter 20 | loss: 0.334471
  LBFGS iter 40 | loss: 0.334282
  Calibrated params: V0=0.068288, theta=0.001000, sigma=0.058807, rho=-0.621414

Evaluating ML surrogate on full dataset...
Evaluating actual stoch-vol pricer on full dataset...

=== ML Surrogate (test set) | HESTON | lbfgs ===
  MSE:  12317.171408  RMSE: 110.982753
  MAPE: 1.378838  R²:   -0.238040

=== Stoch-Vol Pricer (test set) | HESTON | lbfgs ===
  MSE:  89.019327  RMSE: 9.435005
  MAPE: 0.504683  R²:   0.991052
Train: 117,065 (Set A: 58,532 | Set B: 58,533)  |  Test: 46,777
Computing stoch-vol targets for Set A (LHS)...
  Dr

In [ ]:
def one_step_stochvol_ml(
    data,
    model_type: str,
    ml_type:    str,
    train_size: float = 0.7,
    batch_size: int   = 2048,
    n_epochs:   int   = 100,
    patience:   int   = 10,
    params:     dict  = None,
    seed:       int   = 42,
    verbose:    bool  = True,
):

    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _p = params or {}

    if model_type == "heston":
        param_names  = ["V0", "theta", "sigma", "rho"]
        param_bounds = [(1e-3, 0.30), (1e-3, 0.30), (1e-2, 1.0), (-0.999, 0.0)]
        attr_cols    = ["stock_price", "strike_price", "time_to_mat", "rf", "VRP", "cp_flag_one"]
        sv_col       = "price"
        compare_col  = "price"
    else:
        param_names  = ["alphaS", "rhoS", "nuS"]
        param_bounds = [(1e-3, 10.0), (-0.999, -0.001), (1e-3, 10.0)]
        attr_cols    = ["forward_price", "stock_price", "strike_price", "time_to_mat", "rf", "cp_flag_one"]
        sv_col       = "price"
        compare_col  = "price"

    n_params = len(param_names)
    lbs_np   = np.array([b[0] for b in param_bounds])
    ubs_np   = np.array([b[1] for b in param_bounds])

    no_scale_cols = {"cp_flag_one", sv_col}

    def compute_sv_batch(df, params_np, return_ivol=False):
        N = len(df); out = np.zeros(N, dtype=np.float64)
        out_ivol = np.zeros(N, dtype=np.float64) if (model_type == "sabr" and return_ivol) else None
        for s in range(0, N, batch_size):
            e = min(s + batch_size, N)
            def t(arr): return torch.tensor(arr, dtype=torch.float64, device=device)
            if model_type == "heston":
                S0  = t(df["stock_price"].values[s:e]); K  = t(df["strike_price"].values[s:e])
                tau = t(df["time_to_mat"].values[s:e]); rf = t(df["rf"].values[s:e])
                lam = t(df["VRP"].values[s:e]);         cp = t(df["cp_flag_one"].values[s:e])
                V0  = t(params_np[s:e,0]); kap = torch.tensor(2.0, dtype=torch.float64, device=device)
                tht = t(params_np[s:e,1]); sig = t(params_np[s:e,2]); rho = t(params_np[s:e,3])
                call  = heston_price_gpu(S0, K, V0, rf, lam, tau, kap, tht, sig, rho)
                price = torch.where(cp > 0, call, put_heston_gpu(call, S0, K, rf, tau))
                out[s:e] = price.cpu().numpy()
            else:
                F   = t(df["forward_price"].values[s:e]); K  = t(df["strike_price"].values[s:e])
                tau = t(df["time_to_mat"].values[s:e])
                S0  = t(df["stock_price"].values[s:e]); rf = t(df["rf"].values[s:e])
                cp  = t(df["cp_flag_one"].values[s:e])
                alp = t(params_np[s:e,0]); bet = torch.ones(e-s,dtype=torch.float64,device=device)
                rho = t(params_np[s:e,1]); nu  = t(params_np[s:e,2])
                sabr_ivol    = SABR_vol(alp, bet, rho, nu, F, K, tau)
                out[s:e] = BS_pricer(S0, K, rf, sabr_ivol, tau, cp).cpu().numpy()
                if return_ivol:
                    out_ivol[s:e] = sabr_ivol.cpu().numpy()
        if return_ivol:
            return out, out_ivol
        return out

    data = data.copy().reset_index(drop=True)
    if "cp_flag_one" not in data.columns:
        data["cp_flag_one"] = data["cp_flag"].apply(lambda x: 1.0 if x == "C" else -1.0)

    dates  = data["date"].sort_values().unique()
    cutoff = dates[int(len(dates) * train_size)]
    train_mask = data["date"] < cutoff
    train = data[train_mask].copy().reset_index(drop=True)
    test  = data[~train_mask].copy().reset_index(drop=True)
    if verbose: print(f"Train: {len(train):,}  |  Test: {len(test):,}")

    lhs_unit   = LatinHypercube(d=n_params, seed=seed).random(n=len(train))
    params_lhs = np.array([lb+(ub-lb)*lhs_unit[:,j]
                            for j,(lb,ub) in enumerate(param_bounds)]).T
    if verbose: print("Computing stoch-vol outputs for training set (LHS)...")
    sv_train = compute_sv_batch(train, params_lhs)
    valid    = np.isfinite(sv_train) & (sv_train > 0)
    if verbose and (~valid).sum() > 0:
        print(f"  Dropping {(~valid).sum()} non-finite rows")
    train_v  = train[valid].reset_index(drop=True)
    params_v = params_lhs[valid]; sv_v = sv_train[valid]

    X_tr = np.concatenate([
        train_v[attr_cols].values.astype(np.float32),
        np.log(np.clip(sv_v, 1e-8, None)).reshape(-1,1).astype(np.float32),
    ], axis=1)
    Y_tr = params_v.astype(np.float32)

    all_x_cols = attr_cols + [sv_col]
    n_feat = X_tr.shape[1]

    scale_mask = np.array([c not in no_scale_cols for c in all_x_cols])

    scaler_y = StandardScaler().fit(Y_tr)
    Y_tr_sc  = scaler_y.transform(Y_tr).astype(np.float32)

    if ml_type in ("adam", "lbfgs"):
        scaler_x = StandardScaler()
        scaler_x.fit(X_tr[:, scale_mask])
        def _scale_x(X):
            Xs = X.copy()
            Xs[:, scale_mask] = scaler_x.transform(X[:, scale_mask])
            return Xs.astype(np.float32)
        X_tr_sc = _scale_x(X_tr)
    else:
        X_tr_sc = X_tr

    n_val      = max(1, int(len(X_tr_sc) * 0.15))
    n_tr2      = len(X_tr_sc) - n_val
    X_tr2_sc, X_val_sc = X_tr_sc[:n_tr2], X_tr_sc[n_tr2:]
    Y_tr2_sc, Y_val_sc = Y_tr_sc[:n_tr2], Y_tr_sc[n_tr2:]

    if verbose:
        print(f"Features: {n_feat}  |  Targets: {n_params}"
              f"  |  {n_tr2:,} train  {n_val:,} val")

    if ml_type in ("adam", "lbfgs"):
        n_layers = _p.get('n_layers', 3)
        n_units  = _p.get('n_units',  64)

        class _MLP(nn.Module):
            def __init__(self, n_in, n_out):
                super().__init__()
                layers = [nn.Linear(n_in, n_units), nn.GELU()]
                for _ in range(n_layers - 1):
                    layers += [nn.Linear(n_units, n_units), nn.GELU()]
                layers.append(nn.Linear(n_units, n_out))
                self.net = nn.Sequential(*layers)
            def forward(self, x): return self.net(x)

        ann    = _MLP(n_feat, n_params).to(device).double()
        loss_f = nn.MSELoss()
        Xtr2_t = torch.tensor(X_tr2_sc, dtype=torch.float64, device=device)
        Ytr2_t = torch.tensor(Y_tr2_sc, dtype=torch.float64, device=device)
        Xval_t = torch.tensor(X_val_sc, dtype=torch.float64, device=device)
        Yval_t = torch.tensor(Y_val_sc, dtype=torch.float64, device=device)

        best_val   = float('inf')
        best_state = {k: v.clone() for k, v in ann.state_dict().items()}
        no_improve = 0

        if ml_type == "adam":
            opt = optim.Adam(ann.parameters(), lr=1e-3)
            for ep in range(n_epochs):
                ann.train()
                perm = torch.randperm(len(Xtr2_t), device=device)
                for i in range(0, len(Xtr2_t), batch_size):
                    xb = Xtr2_t[perm[i:i+batch_size]]; yb = Ytr2_t[perm[i:i+batch_size]]
                    opt.zero_grad(); loss_f(ann(xb), yb).backward(); opt.step()
                ann.eval()
                with torch.no_grad():
                    val_loss = loss_f(ann(Xval_t), Yval_t).item()
                if val_loss < best_val:
                    best_val = val_loss
                    best_state = {k: v.clone() for k, v in ann.state_dict().items()}
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        if verbose: print(f"  Early stop epoch {ep+1} | best val: {best_val:.6f}")
                        break
                if verbose and (ep+1) % max(1, n_epochs//5) == 0:
                    print(f"  Epoch {ep+1}/{n_epochs} | val MSE: {val_loss:.6f}")

        else:
            opt = optim.LBFGS(ann.parameters(), lr=1.0, max_iter=20, history_size=10)
            for ep in range(n_epochs):
                ann.train()
                def _closure():
                    opt.zero_grad()
                    l = loss_f(ann(Xtr2_t), Ytr2_t)
                    l.backward(); return l
                opt.step(_closure)
                ann.eval()
                with torch.no_grad():
                    val_loss = loss_f(ann(Xval_t), Yval_t).item()
                if val_loss < best_val:
                    best_val = val_loss
                    best_state = {k: v.clone() for k, v in ann.state_dict().items()}
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        if verbose: print(f"  Early stop epoch {ep+1} | best val: {best_val:.6f}")
                        break
                if verbose and (ep+1) % max(1, n_epochs//5) == 0:
                    print(f"  Epoch {ep+1}/{n_epochs} | val MSE: {val_loss:.6f}")

        ann.load_state_dict(best_state)

    else:
        model_xgb = MultiOutputRegressor(
            xgb_lib.XGBRegressor(
                gamma=0.01, learning_rate=0.1, max_depth=_p.get('max_depth', 8),
                n_estimators=_p.get('n_estimators', 200), subsample=1.0, reg_lambda=10,
                n_jobs=-1, random_state=seed,
                device=device.type,
            ),
            n_jobs=1,
        )
        model_xgb.fit(X_tr2_sc, Y_tr2_sc)
        if verbose:
            Y_val_pred = model_xgb.predict(X_val_sc)
            val_mse = mean_squared_error(Y_val_sc, Y_val_pred)
            print(f"  XGBoost fitted | val MSE (scaled): {val_mse:.6f}")

    if verbose: print("Predicting stoch-vol params on full dataset (market prices)...")

    X_full = np.concatenate([
        data[attr_cols].values.astype(np.float32),
        np.log(np.clip(data[sv_col].values, 1e-8, None)).reshape(-1,1).astype(np.float32),
    ], axis=1)

    if ml_type in ("adam", "lbfgs"):
        X_full_sc = _scale_x(X_full)
        ann.eval()
        pred_chunks = []
        with torch.no_grad():
            for s in range(0, len(X_full_sc), batch_size):
                e = min(s + batch_size, len(X_full_sc))
                xb = torch.tensor(X_full_sc[s:e], dtype=torch.float64, device=device)
                pred_chunks.append(ann(xb).cpu().numpy())
        Y_pred_sc = np.concatenate(pred_chunks)
    else:
        Y_pred_sc = model_xgb.predict(X_full)

    Y_pred_raw = scaler_y.inverse_transform(Y_pred_sc.astype(np.float32))
    Y_pred     = np.clip(Y_pred_raw, lbs_np, ubs_np)

    if verbose: print("Evaluating predicted params with actual stoch-vol pricer...")
    _sv_full_res = compute_sv_batch(data, Y_pred, return_ivol=(model_type == "sabr"))
    if model_type == "sabr":
        sv_price_full, sv_ivol_full = _sv_full_res
    else:
        sv_price_full = _sv_full_res
        sv_ivol_full = None

    test_idx = np.where(~train_mask.values)[0]
    y_mkt_test = data.loc[test_idx, compare_col].values.astype(np.float64)
    sv_pred_test = sv_price_full[test_idx]
    ok = np.isfinite(sv_pred_test) & np.isfinite(y_mkt_test)

    mse  = mean_squared_error(y_mkt_test[ok], sv_pred_test[ok])
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_mkt_test[ok], sv_pred_test[ok])
    r2   = r2_score(y_mkt_test[ok], sv_pred_test[ok])

    print(f"\n=== One-Step | {model_type.upper()} | {ml_type} ===")
    print(f"  MSE:  {mse:.6f}  RMSE: {rmse:.6f}")
    print(f"  MAPE: {mape:.6f}  R²:   {r2:.6f}")

    data_out = data.copy()
    data_out["out_of_sample"] = (~train_mask).astype(int).values

    for j, pname in enumerate(param_names):
        data_out[pname] = Y_pred[:, j]
    data_out["params_clipped"] = ((Y_pred_raw < lbs_np) | (Y_pred_raw > ubs_np)).any(axis=1).astype(int)

    if model_type == "heston":
        data_out["heston_price"] = sv_price_full
        data_out["kappa"] = 2.0
        inst_vol = heston_avg_var(
            data_out["V0"], data_out["kappa"], data_out["theta"],
            data_out["time_to_mat"]
        ).cpu().numpy()
        data_out["inst_vol_heston"] = inst_vol
        iv_sv, = implied_vol_newton_gpu_batched(
            [data_out],
            ["heston_price"],
        )
        data_out["heston_volatility"] = iv_sv.numpy()
    else:
        data_out["sabr_price"] = sv_price_full
        data_out["sabr_volatility"] = sv_ivol_full

    data_out["mse_optim"]      = mse
    data_out["rmse_optim"]     = rmse
    data_out["mape_optim"]     = mape
    data_out["rsquared_optim"] = r2

    metrics = dict(mse=mse, rmse=rmse, mape=mape, r2=r2)
    return data_out, metrics

In [ ]:
# XGBoost – Heston
set_seed(42)
out_xgb_os_heston, metrics_xgb_os_heston = one_step_stochvol_ml(
    data, model_type="heston", ml_type="xgboost",
    train_size=0.7, batch_size=2048,
    params=_get_params("one_step_xgb_heston"),
    seed=42, verbose=True,
)

# XGBoost – Sabr
set_seed(42)
out_xgb_os_sabr, metrics_xgb_os_sabr = one_step_stochvol_ml(
    data, model_type="sabr", ml_type="xgboost",
    train_size=0.7, batch_size=2048,
    params=_get_params("one_step_xgb_sabr"),
    seed=42, verbose=True,
)

Train: 117,065  |  Test: 46,777
Computing stoch-vol outputs for training set (LHS)...
  Dropping 1137 non-finite rows
Features: 7  |  Targets: 4  |  98,539 train  17,389 val
  XGBoost fitted | val MSE (scaled): 0.859406
Predicting stoch-vol params on full dataset (market prices)...
Evaluating predicted params with actual stoch-vol pricer...

=== One-Step | HESTON | xgboost ===
  MSE:  249.779115  RMSE: 15.804402
  MAPE: 0.527159  R²:   0.974894
Train: 117,065  |  Test: 46,777
Computing stoch-vol outputs for training set (LHS)...
  Dropping 27017 non-finite rows
Features: 7  |  Targets: 3  |  76,541 train  13,507 val
  XGBoost fitted | val MSE (scaled): 0.749747
Predicting stoch-vol params on full dataset (market prices)...
Evaluating predicted params with actual stoch-vol pricer...

=== One-Step | SABR | xgboost ===
  MSE:  62437.859025  RMSE: 249.875687
  MAPE: 23.261559  R²:   -5.275838


In [ ]:
# ANN LBFGS – Heston
set_seed(42)
out_lbfgs_os_heston, metrics_lbfgs_os_heston = one_step_stochvol_ml(
    data, model_type="heston", ml_type="lbfgs",
    train_size=0.7, batch_size=2048, n_epochs=100,
    params=_get_params("one_step_lbfgs_heston"),
    seed=42, verbose=True,
)

# ANN LBFGS – Sabr
set_seed(42)
out_lbfgs_os_sabr, metrics_lbfgs_os_sabr = one_step_stochvol_ml(
    data, model_type="sabr", ml_type="lbfgs",
    train_size=0.7, batch_size=2048, n_epochs=100,
    params=_get_params("one_step_lbfgs_sabr"),
    seed=42, verbose=True,
)

Train: 117,065  |  Test: 46,777
Computing stoch-vol outputs for training set (LHS)...
  Dropping 1137 non-finite rows
Features: 7  |  Targets: 4  |  98,539 train  17,389 val
  Epoch 20/100 | val MSE: 0.810519
  Epoch 40/100 | val MSE: 0.777764
  Epoch 60/100 | val MSE: 0.769683
  Epoch 80/100 | val MSE: 0.767870
  Epoch 100/100 | val MSE: 0.764122
Predicting stoch-vol params on full dataset (market prices)...
Evaluating predicted params with actual stoch-vol pricer...

=== One-Step | HESTON | lbfgs ===
  MSE:  8.999416  RMSE: 2.999903
  MAPE: 0.211856  R²:   0.999095
Train: 117,065  |  Test: 46,777
Computing stoch-vol outputs for training set (LHS)...
  Dropping 27017 non-finite rows
Features: 7  |  Targets: 3  |  76,541 train  13,507 val
  Epoch 20/100 | val MSE: 0.713962
  Epoch 40/100 | val MSE: 0.699171
  Epoch 60/100 | val MSE: 0.694903
  Epoch 80/100 | val MSE: 0.691930
  Epoch 100/100 | val MSE: 0.689904
Predicting stoch-vol params on full dataset (market prices)...
Evaluating p

In [ ]:
### Export the optimized datasets ###
path_twostep_xgb_h = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\twostep_xgb_heston_2003.dta"
path_twostep_xgb_s = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\twostep_xgb_sabr_2003.dta"
path_twostep_lbfgs_h = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\twostep_lbfgs_heston_2003.dta"
path_twostep_lbfgs_s = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\twostep_lbfgs_sabr_2003.dta"

path_onestep_xgb_h = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\onestep_xgb_heston_2003.dta"
path_onestep_xgb_s = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\onestep_xgb_sabr_2003.dta"
path_onestep_lbfgs_h = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\onestep_lbfgs_heston_2003.dta"
path_onestep_lbfgs_s = r"C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\Stochastic_ML_models\onestep_lbfgs_sabr_2003.dta"



out_xgb_heston.to_stata(path_twostep_xgb_h)
out_xgb_sabr.to_stata(path_twostep_xgb_s)
out_lbfgs_heston.to_stata(path_twostep_lbfgs_h)
out_lbfgs_sabr.to_stata(path_twostep_lbfgs_s)
out_xgb_os_heston.to_stata(path_onestep_xgb_h)
out_xgb_os_sabr.to_stata(path_onestep_xgb_s)
out_lbfgs_os_heston.to_stata(path_onestep_lbfgs_h)
out_lbfgs_os_sabr.to_stata(path_onestep_lbfgs_s)